# Cross-cohort embedding maps: PCA and UMAP, all slides and within cohorts

Where do the cells of the cross-cohort slides sit in embedding space, for the exact features the
cross-cohort classifier was trained on? Do cells group by **cohort** (batch) or by **cell type**,
and how does that look **within one cohort**?

- **Slides & cohorts**: every slide in `configs/train_cross_cohort.yaml`, with its cohort from the
  config's `cohorts` patterns. Tissue, disease and H&E magnification come from `xenium_metadata.csv`.
  The held-out test macro F1 comes from `outputs_classifier/cross_cohort/...` when that run is present.
- **Cells**: `CELLS_PER_SLIDE` random usable cells per slide (mapped to a class, `Unknown` dropped),
  so every slide weighs the same whatever its size. `READOUTS` sets the features:
  - `tokens` is the mean of the 4 corner tokens, the cross-cohort classifier's input.
  - `cls` is the CLS token.
- **Maps**: PCA (top `N_PCS` PCs), then UMAP on those PCs. Each is fitted
  (1) on **all slides together** and (2) **separately within each cohort**.
- **How much each factor explains**: PCR, the share of embedding variance explained by cohort, slide
  and cell type. It uses the same definition as `batch_effects_across_models.ipynb`.
- **Slide-level map**: one point per slide, coloured by held-out macro F1. An optional section overlays
  **each fold's saved classifier** on its held-out cohort.

Run it on the cluster with the cellViT kernel: the embeddings live under `DATASETS_ROOT`, read from
`.env`. The first run reads every slide's h5 file. Random rows touch every block, so 98 slides take
about 15–20 min. It also fits 12 UMAPs. Both the cell sample and the maps are cached under
`notebooks/cached_results/`, so later runs only redraw. Changing any setting below starts a fresh
cache entry.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))


def _load_dotenv(path: Path) -> None:
    """Minimal `.env` loader (KEY=value per line), as in batch_effects_across_models.ipynb."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key, value = key.strip(), value.strip()
        if value:
            os.environ.setdefault(key, value)


_load_dotenv(REPO_ROOT / ".env")

In [ ]:
import fnmatch
import hashlib
import json
import time
import warnings
import zlib
from concurrent.futures import ThreadPoolExecutor

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from omegaconf import OmegaConf
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

from src.python.code_configs import paths
from src.python.code_configs.mappings import mapping_factory
from src.python.utils.loading_functions import _read_rows

try:
    import umap
except ImportError:
    umap = None
    print("umap-learn is not installed (`pip install umap-learn`): UMAP panels fall back to PCA.")

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

## Settings

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs" / "train_cross_cohort.yaml"
MODEL_NAME = "UNI2"
CORRECTION_NAME = "raw"

# Read-out name -> embeddings_{key} datasets averaged into it. "tokens" is the cross-cohort
# classifier's input (loading_functions.load_unique_wsi averages the configured datasets the
# same way). Drop "cls" to halve the first load.
READOUTS = {
    "tokens": ["bottom_left", "bottom_right", "top_left", "top_right"],
    "cls": ["cls"],
}

CELLS_PER_SLIDE = 2_000   # random usable cells per slide (98 slides -> ~196K cells, ~1.2 GB per read-out)
SEED = 42
N_PCS = 50                # PCs kept: the UMAP input and what PCR is computed on
# random_state makes UMAP reproducible but single-threaded; set it to None for a faster, parallel fit.
UMAP_KWARGS = dict(n_neighbors=30, min_dist=0.3, metric="euclidean", random_state=SEED)

# Cached cell samples and maps (gitignored). Delete a sub-folder to force a recompute.
CACHE_DIR = REPO_ROOT / "notebooks" / "cached_results" / "cross_cohort_embedding_maps"
SAVE_FIGS = False         # True: also write every figure as a PNG to FIG_DIR
FIG_DIR = REPO_ROOT / "notebooks" / "figures_cross_cohort_maps"

# Classifier outputs (per-slide test scores) and saved weights of the cross-cohort runs.
OUTPUTS_ROOT = REPO_ROOT / "outputs_classifier"
SAVED_MODELS_ROOT = REPO_ROOT / "saved_models" / "classification_linear"

## Slides, cohorts and per-slide results

One row per slide. `test_fold` is the fold that holds out the slide's cohort. `test_macro_f1_<read-out>`
is that fold's macro F1 on the slide over the classes present in it, the same number as in the HTML
report (`scripts/make_results_report.py`).

In [ ]:
cfg = OmegaConf.load(CONFIG_PATH)
EXPERIMENT = str(cfg.experiment)
MAPPING_NAME = str(cfg.mapping)
MAPPING = mapping_factory(MAPPING_NAME)
CONSIDER_MATCHING = bool(OmegaConf.select(cfg, "data.consider_matching", default=True))
CLASS_NAMES = sorted({c for c in MAPPING.values() if c != "Unknown"})
COHORTS = [str(c) for c in cfg.cohorts]
H5_ROOT = Path(paths.DATASETS_ROOT) / f"{MODEL_NAME}_h5" / CORRECTION_NAME


def side_wsis(side_cfg) -> list[str]:
    return [k for k in side_cfg.keys() if k != "cell_type_proportions"] if side_cfg is not None else []


def wsi_cohort(wsi: str) -> str | None:
    """First cohort, in declaration order, with a matching fnmatch pattern: the rule
    experiment.py's _wsi_cohorts applies (not imported: it pulls in mlflow and hydra)."""
    for cohort, patterns in cfg.cohorts.items():
        if any(fnmatch.fnmatchcase(wsi, str(p)) for p in patterns):
            return str(cohort)
    return None


def results_dir(readout: str) -> Path:
    """Where experiment.py writes a run's split_k/ folders:
    {outputs}/{experiment}/{model}/{correction}/{mapping}/{matching}/{sorted read-out datasets}."""
    return (OUTPUTS_ROOT / EXPERIMENT / MODEL_NAME / CORRECTION_NAME / MAPPING_NAME
            / ("matching" if CONSIDER_MATCHING else "all_cells") / "_".join(sorted(READOUTS[readout])))


def short_name(wsi: str, n: int = 30) -> str:
    for prefix in ("Xeniumranger_V1_", "Xenium_V1_", "Xenium_Prime_", "Xenium_"):
        if wsi.startswith(prefix):
            wsi = wsi[len(prefix):]
            break
    return wsi if len(wsi) <= n else wsi[: n - 1] + "…"


rows = {}
for k, sp in enumerate(cfg.splits):
    for side in ("train", "val", "test"):
        for w in side_wsis(OmegaConf.select(sp, side)):
            row = rows.setdefault(w, {"wsi": w, "cohort": wsi_cohort(w), "test_fold": pd.NA})
            if side == "test":
                row["test_fold"] = k
slides = pd.DataFrame(rows.values()).set_index("wsi").sort_index()
assert slides["cohort"].notna().all(), f"slides matching no cohort pattern: {slides.index[slides['cohort'].isna()].tolist()}"

meta = pd.read_csv(REPO_ROOT / "xenium_metadata.csv", dtype=str)
meta = meta.drop_duplicates("xenium_folder").set_index("xenium_folder")
meta_key = {w: next((k for k in (w, f"{w}_out", f"{w}_outs") if k in meta.index), None) for w in slides.index}
for name, col in {"tissue": "tissue_of_origin", "disease": "disease_state", "he_magnification": "he_magnification"}.items():
    slides[name] = [meta.at[meta_key[w], col] if meta_key[w] else "unknown" for w in slides.index]
slides["tissue"] = slides["tissue"].str.replace(r"\s*\(.*\)\s*$", "", regex=True)

for readout in READOUTS:
    scores = {}
    for f in sorted(results_dir(readout).glob("split_*/per_slide_metrics.csv")):
        df = pd.read_csv(f)
        scores.update(df.loc[df["set"] == "test"].set_index("wsi")["macro_f1_present"].to_dict())
    slides[f"test_macro_f1_{readout}"] = slides.index.map(scores).astype(float)
    print(f"{readout}: held-out test scores for {len(scores)} slides"
          + ("" if scores else f" (nothing under {results_dir(readout)})"))

print(f"{len(slides)} slides, {slides['cohort'].nunique()} cohorts, mapping {MAPPING_NAME} ({', '.join(CLASS_NAMES)})")
(slides.groupby("cohort", sort=False)
       .agg(slides=("tissue", "size"), test_fold=("test_fold", "first"),
            tissues=("tissue", lambda s: ", ".join(f"{t} ({n})" for t, n in s.value_counts().items())),
            **{f"mean test macro F1 ({r})": (f"test_macro_f1_{r}", "mean") for r in READOUTS})
       .reindex(COHORTS).round(3))

## Cell sample

`CELLS_PER_SLIDE` random usable cells per slide. The seed per slide follows
`loading_functions.load_unique_wsi`. Every read-out is built from the same cells, so maps of
`tokens` and `cls` are directly comparable. Rows are read block-wise with `_read_rows`, the classifier
loader's reader. The sample is cached on first load (`float16`, plenty for PCA/UMAP).

In [ ]:
def _cache_key(*parts) -> str:
    return hashlib.sha1(json.dumps(parts, sort_keys=True, default=str).encode()).hexdigest()[:10]


SAMPLE_DIR = CACHE_DIR / (f"{EXPERIMENT}_{MODEL_NAME}_{CORRECTION_NAME}_{CELLS_PER_SLIDE}per_slide_"
                          + _cache_key(MAPPING_NAME, CONSIDER_MATCHING, SEED, READOUTS, list(slides.index)))


def _sample_slide(wsi: str) -> dict:
    """CELLS_PER_SLIDE random usable cells of one slide, with every read-out of those cells."""
    path = H5_ROOT / wsi / "embeddings_dataset.h5"
    if not path.exists():
        return {"wsi": wsi, "error": f"no {path}"}
    with h5py.File(path, "r", locking=False) as f:
        raw = f["cell_labels"][()]
        keep = np.array([MAPPING.get(lab, "Unknown") != "Unknown" for lab in raw], dtype=bool)
        if CONSIDER_MATCHING and "matching" in f:
            keep &= f["matching"][()].astype(bool)
        rows = np.flatnonzero(keep)
        if len(rows) > CELLS_PER_SLIDE:
            rng = np.random.default_rng([SEED, zlib.crc32(wsi.encode())])
            rows = np.sort(rng.choice(rows, size=CELLS_PER_SLIDE, replace=False))
        needed = sorted({k for keys in READOUTS.values() for k in keys if f"embeddings_{k}" in f})
        tokens = {k: _read_rows(f[f"embeddings_{k}"], rows).astype(np.float32) for k in needed}
        cell_ids = [c.decode() if isinstance(c, bytes) else str(c) for c in f["cell_ids"][()][rows]]
    feats = {name: np.mean([tokens[k] for k in keys], axis=0) if all(k in tokens for k in keys) else None
             for name, keys in READOUTS.items()}
    return {"wsi": wsi, "cell_id": np.array(cell_ids), "cell_type": np.array([MAPPING[lab] for lab in raw[rows]]),
            "feats": feats, "n_cells": len(raw), "n_usable": int(keep.sum())}


def load_sample() -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    if (SAMPLE_DIR / "cells.csv").exists():
        cells = pd.read_csv(SAMPLE_DIR / "cells.csv", dtype={"wsi": str, "cell_id": str, "cell_type": str})
        X = {r: np.load(SAMPLE_DIR / f"X_{r}.npy").astype(np.float32) for r in READOUTS if (SAMPLE_DIR / f"X_{r}.npy").exists()}
        print(f"Cached sample {SAMPLE_DIR.name}: {len(cells):,} cells from {cells['wsi'].nunique()} slides")
        return cells, X
    if not H5_ROOT.is_dir():
        raise FileNotFoundError(f"{H5_ROOT} does not exist: DATASETS_ROOT (from .env) must point at the "
                                f"extracted embeddings -- run this notebook on the cluster.")
    t0, parts = time.perf_counter(), []
    with ThreadPoolExecutor(max_workers=8) as pool:
        for i, res in enumerate(pool.map(_sample_slide, slides.index), 1):
            if "error" in res:
                print(f"[{i}/{len(slides)}] {res['wsi']}: SKIPPED, {res['error']}")
                continue
            missing = [r for r, v in res["feats"].items() if v is None]
            print(f"[{i}/{len(slides)}] {res['wsi']}: {len(res['cell_type']):,} of {res['n_usable']:,} usable cells "
                  f"({res['n_cells']:,} in slide)" + (f" -- no {missing} read-out" if missing else ""))
            parts.append(res)
    assert parts, "no slide could be read"
    cells = pd.DataFrame({
        "wsi": np.concatenate([np.full(len(p["cell_type"]), p["wsi"]) for p in parts]),
        "cell_id": np.concatenate([p["cell_id"] for p in parts]),
        "cell_type": np.concatenate([p["cell_type"] for p in parts]),
    })
    X = {}
    for r in READOUTS:
        dims = {p["feats"][r].shape[1] for p in parts if p["feats"][r] is not None}
        if not dims:
            print(f"read-out {r!r}: no slide has it, dropped")
            continue
        assert len(dims) == 1, f"read-out {r!r} has different widths across slides: {dims}"
        d = dims.pop()
        X[r] = np.concatenate([p["feats"][r] if p["feats"][r] is not None else np.full((len(p["cell_type"]), d), np.nan, np.float32)
                               for p in parts])
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    cells.to_csv(SAMPLE_DIR / "cells.csv", index=False)
    for r, x in X.items():
        np.save(SAMPLE_DIR / f"X_{r}.npy", x.astype(np.float16))
    print(f"Sampled {len(cells):,} cells from {len(parts)} slides in {time.perf_counter() - t0:.0f} s -> {SAMPLE_DIR}")
    return cells, X


cells, X = load_sample()
for r, x in X.items():
    cells[f"has_{r}"] = ~np.isnan(x[:, 0])
for col in ("cohort", "tissue", "disease", "he_magnification", "test_fold"):
    cells[col] = cells["wsi"].map(slides[col])
print({r: x.shape for r, x in X.items()})
cells.groupby(["cohort", "cell_type"]).size().unstack(fill_value=0).reindex(index=COHORTS, columns=CLASS_NAMES, fill_value=0)

## Plot helpers

Colours are fixed per entity, in the order validated for colour-vision deficiency on adjacent pairs.
Classes get the same colours as the HTML report. Cohorts reuse the first five hues, and every panel
has its own legend. Past 8 categories (slides, tissues) the plots switch to one small panel per
category: every cell in grey, that category on top. Points are drawn in a shuffled order, so no
category hides the others.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
CLASS_COLORS = dict(zip(CLASS_NAMES, PALETTE))
COHORT_COLORS = dict(zip(COHORTS, PALETTE))
BACKGROUND = "#dcdbd5"   # every other cell, behind a highlighted subset
HIGHLIGHT = "#0b0b0b"
MUTED = "#898781"
# One hue, light = low, for continuous values (the HTML report's F1 ramp); its lightest step stays
# visible on white.
SEQ_CMAP = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])


def point_size(n: int) -> float:
    return float(np.clip(4e4 / max(n, 1), 0.2, 8))


def _clean(ax, title: str = "") -> None:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, fontsize=8.5)


def scatter_by(ax, xy: np.ndarray, labels, colors: dict, title: str = "") -> None:
    """Each category in its colour, with a legend listing its cell count."""
    labels = np.asarray(labels)
    idx = np.random.default_rng(SEED).permutation(len(xy))
    ax.scatter(xy[idx, 0], xy[idx, 1], c=[colors.get(lab, BACKGROUND) for lab in labels[idx]],
               s=point_size(len(xy)), linewidths=0, rasterized=True)
    handles = [Line2D([], [], ls="", marker="o", ms=5, color=c, label=f"{lab} ({(labels == lab).sum():,})")
               for lab, c in colors.items() if (labels == lab).any()]
    ax.legend(handles=handles, fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    _clean(ax, title)


def scatter_values(ax, xy: np.ndarray, values, title: str, label: str) -> None:
    """Cells coloured by a continuous value, with a colour bar."""
    values = np.asarray(values, dtype=float)
    idx = np.random.default_rng(SEED).permutation(len(xy))
    sc = ax.scatter(xy[idx, 0], xy[idx, 1], c=values[idx], cmap=SEQ_CMAP, s=point_size(len(xy)),
                    linewidths=0, rasterized=True)
    ax.figure.colorbar(sc, ax=ax, shrink=0.8, label=label)
    _clean(ax, title)


def _highlight(ax, xy: np.ndarray, mask: np.ndarray, color: str) -> None:
    ax.scatter(xy[~mask, 0], xy[~mask, 1], c=BACKGROUND, s=0.5 * point_size(len(xy)), linewidths=0, rasterized=True)
    ax.scatter(xy[mask, 0], xy[mask, 1], c=color, s=float(np.clip(2e3 / max(mask.sum(), 1), 0.8, 6)),
               linewidths=0, rasterized=True)


def facet_highlight(xy: np.ndarray, labels, order: list, title: str, colors: dict | None = None,
                    titles: dict | None = None, ncols: int = 6, panel: float = 2.1):
    """One panel per category: every cell in grey, that category's cells on top."""
    labels = np.asarray(labels)
    nrows = int(np.ceil(len(order) / ncols))
    ncols = min(ncols, len(order))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel * ncols, panel * nrows + 0.6), squeeze=False)
    for ax, cat in zip(axes.flat, order):
        mask = labels == cat
        _highlight(ax, xy, mask, (colors or {}).get(cat, HIGHLIGHT))
        _clean(ax, f"{(titles or {}).get(cat, cat)}\n{mask.sum():,} cells")
    for ax in axes.flat[len(order):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=11)
    finish(fig)
    return fig


def cohort_class_grid(xy: np.ndarray, sub: pd.DataFrame, title: str, panel: float = 2.0):
    """Rows = cohorts, columns = cell types: does one cell type land in the same place whatever
    its cohort? Each panel highlights that (cohort, cell type) over every cell in grey."""
    fig, axes = plt.subplots(len(COHORTS), len(CLASS_NAMES), squeeze=False,
                             figsize=(panel * len(CLASS_NAMES) + 0.6, panel * len(COHORTS) + 0.6))
    coh, cls = sub["cohort"].to_numpy(), sub["cell_type"].to_numpy()
    for i, cohort in enumerate(COHORTS):
        for j, name in enumerate(CLASS_NAMES):
            ax, mask = axes[i, j], (coh == cohort) & (cls == name)
            _highlight(ax, xy, mask, CLASS_COLORS[name])
            if not mask.any():
                ax.text(0.5, 0.5, "none", transform=ax.transAxes, ha="center", va="center", color=MUTED)
            _clean(ax, (f"{name}\n" if i == 0 else "") + f"{mask.sum():,} cells")
            if j == 0:
                ax.set_ylabel(cohort, fontsize=10, rotation=0, ha="right", va="center")
    fig.suptitle(title, fontsize=11)
    finish(fig)
    return fig


def finish(fig) -> None:
    """tight_layout, keeping room at the top for the figure's suptitle (up to two lines)."""
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.35 / fig.get_figheight() if fig.get_suptitle() else 1))


def show(fig, name: str) -> None:
    if SAVE_FIGS:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

## PCA and UMAP: all slides, then each cohort on its own

`MAPS[read-out, scope]` holds the cells' row indices into `cells` (`idx`), their top `N_PCS` PCs
(`pcs`, `var_ratio`) and the UMAP of those PCs (`umap`). `scope` is `"all"` or a cohort name. A
cohort's maps are **refitted on that cohort's cells only**, so its PCs describe the variation inside
the cohort rather than how it differs from the other cohorts.

In [ ]:
MAP_DIR = SAMPLE_DIR / f"maps_{_cache_key(N_PCS, UMAP_KWARGS)}"


def embed(readout: str, scope: str, idx: np.ndarray) -> dict:
    """PCA (N_PCS components) of X[readout][idx], then UMAP on those PCs; cached on disk."""
    path = MAP_DIR / f"{readout}__{scope}.npz"
    if path.exists():
        z = np.load(path)
        return {"pcs": z["pcs"], "var_ratio": z["var_ratio"], "umap": z["umap"] if z["umap"].size else None}
    t0, x = time.perf_counter(), X[readout][idx]
    pca = PCA(n_components=min(N_PCS, x.shape[1], len(x) - 1), random_state=SEED).fit(x)
    pcs = pca.transform(x).astype(np.float32)
    um = None
    if umap is not None and len(x) > UMAP_KWARGS["n_neighbors"]:
        with warnings.catch_warnings():  # "n_jobs overridden by setting random_state": expected with a seed
            warnings.filterwarnings("ignore", message="n_jobs value")
            um = umap.UMAP(**UMAP_KWARGS).fit_transform(pcs).astype(np.float32)
    MAP_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(path, pcs=pcs, var_ratio=pca.explained_variance_ratio_,
             umap=um if um is not None else np.empty((0, 2), np.float32))
    print(f"  {readout} / {scope}: {len(x):,} cells, PCA + UMAP in {time.perf_counter() - t0:.0f} s")
    return {"pcs": pcs, "var_ratio": pca.explained_variance_ratio_, "umap": um}


def map_xy(m: dict) -> tuple[np.ndarray, str]:
    """The 2-D layout to draw: UMAP when available, else PC1/PC2."""
    return (m["umap"], "UMAP") if m["umap"] is not None else (m["pcs"][:, :2], "PCA")


SCOPES = ["all"] + COHORTS
MAPS = {}
for readout in X:
    has = cells[f"has_{readout}"].to_numpy(bool)
    for scope in SCOPES:
        idx = np.flatnonzero(has if scope == "all" else has & (cells["cohort"] == scope).to_numpy())
        if len(idx) < 3:
            print(f"  {readout} / {scope}: only {len(idx)} cells, skipped")
            continue
        MAPS[readout, scope] = {"idx": idx, **embed(readout, scope, idx)}
print(f"{len(MAPS)} maps ready in {MAP_DIR}")

### All slides together

The overview row shows PCA and UMAP coloured by cohort and by cell type. Then:
- one panel per cohort;
- a cohort × cell-type grid: if a cell type sits in the same region in every row, the read-out
  transfers across cohorts; if every row sits somewhere else, the cohort dominates;
- one panel per tissue.

In [ ]:
for readout in X:
    m = MAPS.get((readout, "all"))
    if m is None:
        continue
    sub = cells.iloc[m["idx"]]
    xy, kind = map_xy(m)
    v = m["var_ratio"]
    head = f"{MODEL_NAME} · {readout} · all {sub['wsi'].nunique()} slides, {len(sub):,} cells"

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
    scatter_by(axes[0], m["pcs"][:, :2], sub["cohort"], COHORT_COLORS, f"PCA by cohort (PC1 {v[0]:.1%}, PC2 {v[1]:.1%})")
    scatter_by(axes[1], xy, sub["cohort"], COHORT_COLORS, f"{kind} by cohort")
    scatter_by(axes[2], xy, sub["cell_type"], CLASS_COLORS, f"{kind} by cell type")
    fig.suptitle(head, fontsize=11)
    finish(fig)
    show(fig, f"all_{readout}_overview")

    show(facet_highlight(xy, sub["cohort"], COHORTS, f"{head}\n{kind}, one cohort at a time",
                         colors=COHORT_COLORS, ncols=len(COHORTS), panel=2.8), f"all_{readout}_by_cohort")
    show(cohort_class_grid(xy, sub, f"{head}\n{kind}: cohort (rows) × cell type (columns)"),
         f"all_{readout}_cohort_x_cell_type")
    tissue_slides = sub.groupby("tissue")["wsi"].nunique()
    tissues = sub["tissue"].value_counts().index.tolist()
    show(facet_highlight(xy, sub["tissue"], tissues, f"{head}\n{kind}, one tissue at a time",
                         titles={t: f"{t} ({tissue_slides[t]} slides)" for t in tissues}, ncols=7, panel=1.9),
         f"all_{readout}_by_tissue")

### Within each cohort

For each cohort, PCA and UMAP are refitted on that cohort's cells. The overview row colours them by
cell type, then by slide when there are at most 8 slides. Otherwise the third panel uses tissue when
it varies, else each slide's held-out test macro F1. Then comes one panel per slide, ordered by
held-out test macro F1 (best first), with the slide's tissue and score in its title.

In [ ]:
def third_panel(sub: pd.DataFrame, readout: str) -> tuple[str, pd.Series]:
    """What colours the third overview panel: the slides when they fit the palette, else the tissue
    when it varies, else each slide's held-out test macro F1 (continuous), else the disease state."""
    if sub["wsi"].nunique() <= len(PALETTE):
        return "slide", sub["wsi"].map(short_name)
    if 1 < sub["tissue"].nunique() <= len(PALETTE):
        return "tissue", sub["tissue"]
    f1 = sub["wsi"].map(slides[f"test_macro_f1_{readout}"])
    if f1.notna().any():
        return "the slide's held-out test macro F1", f1
    return "disease", sub["disease"]


for readout in X:
    for cohort in COHORTS:
        m = MAPS.get((readout, cohort))
        if m is None:
            continue
        sub = cells.iloc[m["idx"]]
        xy, kind = map_xy(m)
        v = m["var_ratio"]
        head = (f"{MODEL_NAME} · {readout} · {cohort} only: {sub['wsi'].nunique()} slides, {len(sub):,} cells "
                f"(PCA and {kind} fitted on this cohort)")

        fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
        scatter_by(axes[0], m["pcs"][:, :2], sub["cell_type"], CLASS_COLORS, f"PCA by cell type (PC1 {v[0]:.1%}, PC2 {v[1]:.1%})")
        scatter_by(axes[1], xy, sub["cell_type"], CLASS_COLORS, f"{kind} by cell type")
        by, values = third_panel(sub, readout)
        if by.endswith("macro F1"):
            scatter_values(axes[2], xy, values, f"{kind} by {by}", "test macro F1")
        else:
            scatter_by(axes[2], xy, values, dict(zip(sorted(values.unique()), PALETTE)), f"{kind} by {by}")
        fig.suptitle(head, fontsize=11)
        finish(fig)
        show(fig, f"cohort_{cohort}_{readout}_overview")

        info = slides.loc[sub["wsi"].unique()]
        f1 = info[f"test_macro_f1_{readout}"]
        order = f1.sort_values(ascending=False, na_position="last").index.tolist()
        titles = {w: f"{short_name(w, 26)}\n{info.at[w, 'tissue']}" + ("" if pd.isna(f1[w]) else f" · test F1 {f1[w]:.2f}")
                  for w in order}
        show(facet_highlight(xy, sub["wsi"], order, f"{head}\n{kind}, one slide at a time", titles=titles, ncols=6),
             f"cohort_{cohort}_{readout}_by_slide")

## How much does each factor explain?

PCR (principal component regression, scIB's definition, as in `batch_effects_across_models.ipynb`):
regress each of the map's top PCs on one covariate (one-hot), then sum the R² weighted by each PC's
share of the total variance. The result is the fraction of the embedding's variance that the covariate
explains.
- A high **cohort** or **slide** value means a strong batch component.
- A high **cell type** value means the biology is what the embedding mostly encodes.
- Slides sit inside cohorts, so on all slides PCR(slide) ≥ PCR(cohort). The difference is the
  slide-to-slide variation within cohorts.
- Within a cohort, only slide and cell type apply.

In [ ]:
def pcr(pcs: np.ndarray, var_ratio: np.ndarray, covariate) -> float:
    dummies = pd.get_dummies(pd.Series(np.asarray(covariate)).astype(str)).to_numpy(dtype=np.float32)
    if dummies.shape[1] <= 1:
        return np.nan
    pred = LinearRegression().fit(dummies, pcs).predict(dummies)
    r2 = 1 - ((pcs - pred) ** 2).sum(0) / ((pcs - pcs.mean(0)) ** 2).sum(0)
    return float(np.sum(var_ratio[: pcs.shape[1]] * np.clip(r2, 0, None)))


COVARIATES = {"cohort": "cohort", "slide": "wsi", "cell type": "cell_type"}
pcr_rows = []
for (readout, scope), m in MAPS.items():
    sub = cells.iloc[m["idx"]]
    row = {"read-out": readout, "scope": scope, "slides": sub["wsi"].nunique(), "cells": len(sub),
           f"variance in top {N_PCS} PCs": float(m["var_ratio"].sum())}
    for name, col in COVARIATES.items():
        row[name] = pcr(m["pcs"], m["var_ratio"], sub[col]) if (scope == "all" or name != "cohort") else np.nan
    pcr_rows.append(row)
pcr_df = pd.DataFrame(pcr_rows).set_index(["read-out", "scope"])

fig, axes = plt.subplots(1, len(X), figsize=(6.5 * len(X), 3.8), squeeze=False, sharey=True)
for ax, readout in zip(axes[0], X):
    d = pcr_df.loc[readout].reindex([s for s in SCOPES if (readout, s) in MAPS])
    pos, width = np.arange(len(d)), 0.26
    for i, name in enumerate(COVARIATES):
        ax.bar(pos + (i - 1) * width, d[name].fillna(0), width=width * 0.92, color=PALETTE[i], label=name)
    ax.set_xticks(pos, d.index)
    ax.set_title(f"{readout}: variance explained (PCR)", fontsize=10)
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
axes[0, 0].set_ylabel("share of embedding variance")
axes[0, -1].legend(frameon=False, fontsize=8)
fig.tight_layout()
show(fig, "pcr")
pcr_df.round(3)

## Slide-level map

One point per slide: its mean position in the all-slides PCA (`N_PCS` PCs), projected again to 2-D.
The left panel colours slides by cohort, the right one by held-out test macro F1. The six lowest
slides are numbered on the plot and listed in the table below it. A slide's centroid also reflects its cell-type mix. Set `CENTROID_CLASS` to one
class (e.g. `"Stromal"`) to compare slides on that class only.

In [ ]:
CENTROID_CLASS = None

lowest = []
for readout in X:
    m = MAPS.get((readout, "all"))
    if m is None:
        continue
    sub = cells.iloc[m["idx"]]
    keep = np.ones(len(sub), bool) if CENTROID_CLASS is None else (sub["cell_type"] == CENTROID_CLASS).to_numpy()
    cent = pd.DataFrame(m["pcs"][keep]).groupby(sub["wsi"].to_numpy()[keep]).mean()
    xy = PCA(n_components=2, random_state=SEED).fit_transform(cent.to_numpy())
    info = slides.loc[cent.index]
    f1 = info[f"test_macro_f1_{readout}"].to_numpy(float)
    has_f1 = np.isfinite(f1).any()
    what = "all cells" if CENTROID_CLASS is None else f"{CENTROID_CLASS} cells only"

    fig, axes = plt.subplots(1, 2 if has_f1 else 1, figsize=(13 if has_f1 else 6.5, 5), squeeze=False)
    ax = axes[0, 0]
    for cohort in COHORTS:
        sel = (info["cohort"] == cohort).to_numpy()
        ax.scatter(xy[sel, 0], xy[sel, 1], s=40, color=COHORT_COLORS[cohort], edgecolors="white", linewidths=0.8,
                   label=f"{cohort} ({sel.sum()})")
    ax.legend(frameon=False, fontsize=8)
    _clean(ax, f"Slide centroids ({what}) by cohort")
    if has_f1:
        ax = axes[0, 1]
        sc = ax.scatter(xy[:, 0], xy[:, 1], c=f1, cmap=SEQ_CMAP, vmin=0, vmax=np.nanmax(f1), s=40,
                        edgecolors="#52514e", linewidths=0.5)
        fig.colorbar(sc, ax=ax, shrink=0.8, label="held-out test macro F1 (classes present)")
        low = [i for i in np.argsort(np.where(np.isfinite(f1), f1, np.inf))[:6] if np.isfinite(f1[i])]
        for rank, i in enumerate(low, 1):
            ax.annotate(str(rank), xy[i], xytext=(4, 3), textcoords="offset points", fontsize=8, fontweight="bold")
        lowest.append(info.iloc[low][["cohort", "tissue", "disease"]].rename_axis("wsi").reset_index()
                      .assign(**{"read-out": readout, "rank": range(1, len(low) + 1), "test macro F1": f1[low]}))
        _clean(ax, f"Slide centroids ({what}) by held-out test macro F1 (1 = lowest, listed below)")
    fig.suptitle(f"{MODEL_NAME} · {readout} · {len(cent)} slides", fontsize=11)
    finish(fig)
    show(fig, f"slides_{readout}_centroids")
pd.concat(lowest).set_index(["read-out", "rank"]).round(3) if lowest else None

## Optional: each fold's classifier on its held-out cohort

For each fold, the saved classifier is applied to the sampled cells of the cohort it never saw, and
drawn on that cohort's own map: true class, predicted class, and the misclassified cells. Training
saved these classifiers under `saved_models/` (`save_model` in the config). The path recorded in the
fold's `results_summary.yaml` is used, re-rooted on `SAVED_MODELS_ROOT` when a copy is there. Folds
without weights are skipped.

The table checks the plumbing: accuracy on the sampled cells (mean over slides) should be close to
the slide-mean accuracy in `per_slide_metrics.csv`. If they differ by more than sampling noise, the
features here are not what the classifier saw.

In [ ]:
OVERLAY_READOUT = "tokens"   # read-out the saved classifiers were trained on

try:
    from src.python.classifier_training.saved_models import load_classifier
except ImportError as e:  # torch missing in this kernel
    load_classifier = None
    print(f"Skipping the classifier overlay: {e}")


def fold_model_dir(readout: str, k: int) -> Path | None:
    summary = results_dir(readout) / f"split_{k}" / "results_summary.yaml"
    if not summary.exists():
        return None
    saved = (yaml.safe_load(summary.read_text()).get("provenance") or {}).get("saved_model_dir")
    if not saved:
        return None
    local = SAVED_MODELS_ROOT.joinpath(*Path(saved).parts[-3:])   # {experiment}/{train_cohorts}/{job_name}
    return next((d for d in (local, Path(saved)) if (d / "model.pt").exists()), None)


overlays, check_rows = [], []
for k in sorted(slides["test_fold"].dropna().astype(int).unique()) if load_classifier and OVERLAY_READOUT in X else []:
    cohort = slides.loc[slides["test_fold"] == k, "cohort"].iloc[0]
    m, model_dir = MAPS.get((OVERLAY_READOUT, cohort)), fold_model_dir(OVERLAY_READOUT, k)
    if m is None or model_dir is None:
        print(f"fold {k} ({cohort}): no " + ("map" if m is None else f"saved classifier for {OVERLAY_READOUT}") + ", skipped")
        continue
    clf = load_classifier(model_dir)
    trained_on = sorted(clf.model_info.get("embeddings_datasets") or [])
    if trained_on != sorted(READOUTS[OVERLAY_READOUT]):
        print(f"fold {k}: classifier trained on {trained_on}, not {OVERLAY_READOUT}={READOUTS[OVERLAY_READOUT]}, skipped")
        continue
    sub = cells.iloc[m["idx"]]
    pred = np.asarray(clf.class_names)[clf.predict(X[OVERLAY_READOUT][m["idx"]])]
    correct = pred == sub["cell_type"].to_numpy()
    overlays.append((k, cohort, m, sub, pred, correct))
    per_slide = pd.read_csv(results_dir(OVERLAY_READOUT) / f"split_{k}" / "per_slide_metrics.csv")
    check_rows.append({
        "fold": k, "held-out cohort": cohort, "slides": sub["wsi"].nunique(), "sampled cells": len(sub),
        "accuracy, sampled cells (slide mean)": pd.Series(correct).groupby(sub["wsi"].to_numpy()).mean().mean(),
        "accuracy, per_slide_metrics.csv (slide mean)": per_slide.loc[per_slide["set"] == "test", "accuracy"].mean(),
        "model": model_dir.name,
    })

if overlays:
    fig, axes = plt.subplots(len(overlays), 3, figsize=(17, 4.3 * len(overlays)), squeeze=False)
    for row, (k, cohort, m, sub, pred, correct) in zip(axes, overlays):
        xy, kind = map_xy(m)
        scatter_by(row[0], xy, sub["cell_type"], CLASS_COLORS, f"fold {k} · {cohort} · {kind} · true class")
        scatter_by(row[1], xy, pred, CLASS_COLORS, "predicted class")
        _highlight(row[2], xy, ~correct, HIGHLIGHT)
        _clean(row[2], f"misclassified: {(~correct).sum():,} of {len(correct):,} ({(~correct).mean():.0%})")
    fig.suptitle(f"{MODEL_NAME} · {OVERLAY_READOUT}: each fold's classifier on the cohort it never saw "
                 f"(maps fitted within each cohort)", fontsize=11)
    finish(fig)
    show(fig, f"classifier_overlay_{OVERLAY_READOUT}")
pd.DataFrame(check_rows).round(3) if check_rows else print("No saved classifiers found: nothing to overlay.")

## Extending

- **Another model or correction**: change `MODEL_NAME` / `CORRECTION_NAME`. The slide list and
  cohorts still come from the config; per-slide scores appear once that run's outputs exist.
- **Other read-outs**: add entries to `READOUTS` (e.g. `"top_left": ["top_left"]`). The first run
  after that re-reads the h5 files.
- **More cells or other UMAP settings**: `CELLS_PER_SLIDE`, `N_PCS`, `UMAP_KWARGS`. Each combination
  gets its own cache folder under `CACHE_DIR`; delete old ones to free space.
- **Figures for the paper**: set `SAVE_FIGS = True` to write every figure as a PNG to `FIG_DIR`.